Author: Jessica Xu

# Week 1 — Data Mining, Similarity, and Data Representations

## Project context

This notebook uses a corpus of **4,244 World Cup-related articles** from **NYT-main, the New York Post, and The Athletic**, collected in **July–August 2026** and covering the **2022 and 2026 FIFA World Cups** (2022: ~1,442; 2026: ~2,802).

**Main RQ:** How do NYT-main, the New York Post, and The Athletic differ in the **composition and framing** of their coverage of the 2022 and 2026 FIFA World Cups?

- **SQ1 — Composition:** What events, issues, genres, and jurisdictions receive attention?
- **SQ2 — Framing:** When outlets cover comparable issues or events, how do they differ in problem definition, responsibility attribution, evaluation, treatment/response, and affected actors?

**Social science significance:** By comparing what different news outlets choose to cover and how they describe the same events, this project examines how media priorities and framing shape public understanding and reflect different media cultures around a major global event.

**Week 1:** Test how representation, similarity metric, and candidate-window choices change which cross-outlet articles are treated as comparable. It compares headline Jaccard, headline term-frequency cosine, opening-text cosine, entity overlap, and publication-window restrictions.

**Evaluation design:** Purposeful anchors provide a **known-positive retrieval diagnostic** when a same-event counterpart is known to exist; stratified-random anchors provide a more naturalistic robustness check and are not interpreted as retrieval accuracy because counterpart existence is not verified for every random group.

### Current scholarly usage

Recent research demonstrates that similarity is not a single fixed property of texts; rather, its substantive meaning depends on how observations are represented, which observations are compared, and which similarity measure is used. Eisele, Tolochko, and Boomgaarden (2022), for example, construct a multidimensional representation of political crisis communication based on four theoretically defined dimensions: accessibility, allay, accommodation, and political alignment. They then use cosine similarity to assess whether communication is structurally similar across these dimensions. Importantly, their measure is not intended to capture whether texts use semantically similar language, but whether they have similar profiles on the theoretically meaningful dimensions. This demonstrates how similarity can be used as a measurement strategy rather than simply a measure of lexical resemblance.

Dujeancourt and Garz (2023) use similarity for a different purpose: identifying corresponding news stories across Twitter and Facebook. They represent posts as term-frequency vectors and calculate cosine similarity only among posts published by the same newspaper on the same day, using the maximum cosine similarity to identify likely cross-platform counterparts. Their approach illustrates that defining the appropriate comparison set can be as important as choosing the similarity metric itself. They also considered embedding-based alternatives, including BERT, but found through spot checks that term-frequency cosine similarity performed better for their specific matching task.

Vonk, Bos, and van Sebille (2024) further demonstrate why multiple similarity measures may capture different relationships between texts. To measure textual overlap between scientific press releases and resulting newspaper articles, they combine the Jaccard index, cosine similarity based on term-frequency vectors, and normalized Levenshtein distance. Jaccard captures overlap in the words used, cosine captures similarities in word frequency and distribution, while edit distance captures additions, deletions, and substitutions. They also acknowledge that these traditional measures do not capture deeper semantic relationships.

Taken together, these studies show that similarity is not a single generic measure but a methodological choice that depends on how texts are represented, which observations are compared, and what kind of relationship the researcher wants to capture. For my project comparing World Cup coverage across The New York Times, the New York Post, and The Athletic, I therefore test how alternative representations and similarity measures change which cross-outlet articles appear comparable. I compare lexical Jaccard similarity, term-frequency cosine similarity, and named-entity overlap, while also varying the temporal window within which candidate articles can be matched. I then manually evaluate whether retrieved neighbors describe the same event, the same broader issue, or unrelated content. This helps determine which forms of similarity are useful for later cross-outlet comparisons without assuming that lexical similarity, event similarity, and framing similarity are equivalent.

**References**

Dujeancourt, E., & Garz, M. (2023). The effects of algorithmic content selection on user engagement with news on Twitter. *The Information Society, 39*(5), 263–281.

Eisele, O., Tolochko, P., & Boomgaarden, H. G. (2022). How do executives communicate about crises? A framework for comparative analysis. *European Journal of Political Research, 61*, 952–972.

Vonk, A. N., Bos, M., & van Sebille, E. (2024). Journalism Versus Churnalism: How News Factors in Press Releases Affect Journalistic Processing of Ocean Plastic Research in Newspapers Globally. *Journalism Studies, 25*(16), 2031–2050.

### Setup


In [1]:
import os
import re
from collections import Counter

import pandas as pd
import spacy
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


In [2]:
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
INPUT_FILE = os.path.join(DATA_DIR, "worldcup_method_corpus.parquet")
OUTPUT_DIR = os.path.join(BASE_DIR, "data", "similarity_outputs")
LABELED_FILE = os.path.join(OUTPUT_DIR, "similarity_pairs_labeled.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOWS = [1, 3, 7]
COMPARISON_WINDOW = 3 # Fixed window used when comparing different similarity specifications
ENTITY_TYPES = {"PERSON", "ORG", "GPE", "LOC", "NORP", "FAC", "EVENT"}

# Normalize common U.S. references to the same entity
ENTITY_ALIASES = {"us": "usa", "u.s.": "usa", "u.s": "usa", "united states": "usa", "america": "usa", "american": "usa", "americans": "usa", "usmnt": "usa"}

# Remove outlet names from entity overlap
OUTLET_ENTITIES = {"athletic", "new york times", "times", "post", "new york post"}
GENERIC_TERMS = {"world", "cup", "fifa", "2022", "2026"}
TOP_K = 3
RANDOM_STATE = 40123
TOURNAMENT_START = {2022: pd.Timestamp("2022-11-20"), 2026: pd.Timestamp("2026-06-11")}


### Load corpus

In [3]:
df = pd.read_parquet(INPUT_FILE)
df["pub_date"] = pd.to_datetime(df["pub_date"], errors="coerce")
df["edition"] = pd.to_numeric(df["edition"], errors="coerce").astype("Int64")

print("Rows, columns:", df.shape)
print()
print(pd.crosstab(df["outlet"], df["edition"]))


Rows, columns: (4244, 25)

edition        2022  2026
outlet                   
NYT-main        186   245
New York Post   217   886
The Athletic   1039  1671


In [4]:
# Check how U.S.-related terms appear after headline preprocessing
toks = df["headline_lexical"].str.split().explode()
print("usa:", toks.eq("usa").sum(), "| us:", toks.eq("us").sum(), "| 'u s':", df["headline_lexical"].str.contains(r"\bu s\b").sum())

usa: 290 | us: 32 | 'u s': 0


In [5]:
required_cols = ["article_id", "outlet", "edition", "pub_date", "headline_text", "headline_lexical", "opening_250"]
missing_cols = [col for col in required_cols if col not in df.columns]

assert not missing_cols, f"Missing required columns: {missing_cols}"
assert {"body_text_analysis", "media_blocks_removed"}.issubset(df.columns)
assert df["article_id"].is_unique
assert df["pub_date"].notna().all()
assert df["headline_lexical"].fillna("").ne("").all()
assert df["headline_lexical"].str.split().explode().eq("usa").sum() > 200

print("Input checks passed.")

Input checks passed.


I use headlines and the first 250 words rather than full article text because the matching task focuses on the article's primary event or focal development. The opening 250 words add context beyond the headline while reducing the influence of later background material and differences in article length.

### Prepare Spec A headline tokens


In [6]:
df["headline_tokens"] = df["headline_lexical"].map(lambda text: [token for token in str(text).split() if token not in ENGLISH_STOP_WORDS])
headline_sets = df.set_index("article_id")["headline_tokens"].map(set).to_dict()

print("Empty token sets:", df["headline_tokens"].map(len).eq(0).sum())
print()
print(pd.Series([token for tokens in df["headline_tokens"] for token in tokens]).value_counts().head(20))


Empty token sets: 0

world        3286
cup          3255
usmnt         552
england       360
usa           290
fifa          281
fans          209
argentina     195
soccer        193
messi         190
vs            190
team          180
2026          174
squad         174
qatar         162
final         158
france        151
game          151
win           151
2022          145
Name: count, dtype: int64


### Build anchor sample


In [7]:
df["period"] = df.apply(lambda row: "tournament" if row["pub_date"] >= TOURNAMENT_START[int(row["edition"])] else "pre", axis=1)
print(pd.crosstab([df["edition"], df["outlet"]], df["period"]))


period                 pre  tournament
edition outlet                        
2022    NYT-main        28         158
        New York Post   30         187
        The Athletic   421         618
2026    NYT-main        58         187
        New York Post  193         693
        The Athletic   525        1146


In [8]:
# Known cross-outlet events chosen before retrieval; anchor outlet rotated across events
PURPOSEFUL_IDS = [
    "NYT_nyt://article/8bea560c-aa79-5c2a-bd60-7c0a331bc55d",  # 2022 Qatar bans beer sales at stadiums
    "NYP_nyp_20221121_5d07e10b6b",  # 2022 OneLove armband abandoned under FIFA threat
    "ATH_3940030",  # 2022 U.S. Soccer alters Iran flag on social media
    "ATH_7205423",  # 2026 NJ Transit $150 World Cup train tickets
    "NYT_nyt://article/ade9c476-fd38-5872-b87d-2d5a7db1ad07",  # 2026 Iran team visas, staff denied
    "NYP_nyp_20260612_159a45c031",  # 2026 USMNT beat Paraguay in opener
    "ATH_7354492",  # 2026 Thomas Partey denied entry to Canada
    "NYT_nyt://article/e49d6023-9613-5137-bb18-7ebd022462ff",  # 2026 Trump presents trophy at final
]
purposeful_anchors = df[df["article_id"].isin(PURPOSEFUL_IDS)].assign(anchor_type="purposeful")

assert set(PURPOSEFUL_IDS).issubset(set(df["article_id"]))
print("Purposeful anchors:", len(purposeful_anchors))
purposeful_anchors[["article_id", "outlet", "edition", "pub_date", "headline_text"]]


Purposeful anchors: 8


,article_id,outlet,edition,pub_date,headline_text
23,NYT_nyt://article/8bea560c-aa79-5c2a-bd60-7c0a...,NYT-main,2022,2022-11-18,Ban on Beer Is Latest Flash Point in World Cup...
228,NYP_nyp_20221121_5d07e10b6b,New York Post,2022,2022-11-21,FIFA threats force World Cup teams to abandon ...
1030,ATH_3940030,The Athletic,2022,2022-11-27,U.S. Soccer take down social media posts after...
1479,NYT_nyt://article/ade9c476-fd38-5872-b87d-2d5a...,NYT-main,2026,2026-06-05,Iran’s Soccer Team Allowed Into U.S. for World...
1685,NYT_nyt://article/e49d6023-9613-5137-bb18-7ebd...,NYT-main,2026,2026-07-19,Trump Relents From Spain-Bashing to Present It...
1903,NYP_nyp_20260612_159a45c031,New York Post,2026,2026-06-12,USMNT met the World Cup moment in dominant win...
2595,ATH_7205423,The Athletic,2026,2026-04-17,MetLife World Cup train tickets confirmed at $...
3161,ATH_7354492,The Athletic,2026,2026-06-12,"Thomas Partey denied entry to Canada, unable t..."


Purposeful anchors were selected from widely reported World Cup events before running the retrieval analysis. For each anchor, I searched the other two outlets within ±3 days and manually verified whether a same-event counterpart existed. Because retrieval is evaluated separately by target outlet, the 8 anchors define 16 anchor–target groups; 12 were confirmed to contain a same-event counterpart and are treated as the known-positive evaluation set.

Purposeful anchors are diagnostic rather than representative: they test whether the similarity method can recover a known counterpart when one exists

| Anchor | Event | Confirmed counterpart(s) |
|---|---|---|
| NYT, 2022-11-18 | Qatar bans beer sales at stadiums | NYP: beer ban; Athletic: beer ban |
| NYP, 2022-11-21 | OneLove armband abandoned under FIFA threat | Athletic: armband controversy |
| Athletic, 2022-11-27 | U.S. Soccer alters Iran flag | NYP: altered Iran flag posts; NYT: Iran calls for U.S. expulsion |
| Athletic, 2026-04-17 | NJ Transit $150 fare confirmed | NYP: $150 fare announcement; NYT: $150 transit pricing |
| NYT, 2026-06-05 | Iran team visas; staff denied | Athletic: Iran visa decision |
| NYP, 2026-06-12 | USMNT beats Paraguay in opener | Athletic: USMNT–Paraguay opener |
| Athletic, 2026-06-12 | Thomas Partey denied Canada entry | NYP: Partey denied entry |
| NYT, 2026-07-19 | Trump presents trophy at final | NYP: Trump awards Spain medals; Athletic: Trump at trophy presentation |

In [9]:
random_anchors = df.groupby(["edition", "outlet", "period"], group_keys=False).sample(n=1, random_state=RANDOM_STATE).assign(anchor_type="random") # Sample one random anchor from each edition × outlet × period group
anchors = pd.concat([purposeful_anchors, random_anchors], ignore_index=True) if len(purposeful_anchors) else random_anchors.copy() # combine purposeful and random anchors
anchors = anchors.drop_duplicates("article_id").sort_values(["edition", "outlet", "pub_date"]).reset_index(drop=True)

print("Total anchors:", len(anchors))
print(anchors["anchor_type"].value_counts())
anchors[["article_id", "anchor_type", "outlet", "edition", "period", "pub_date", "headline_text"]]

Total anchors: 20
anchor_type
random        12
purposeful     8
Name: count, dtype: int64


,article_id,anchor_type,outlet,edition,period,pub_date,headline_text
0,NYT_nyt://article/7f1bdf17-c28a-5874-9f60-636c...,random,NYT-main,2022,pre,2022-09-22,Pressing World Cup Question for U.S.: Who’ll S...
1,NYT_nyt://article/8bea560c-aa79-5c2a-bd60-7c0a...,purposeful,NYT-main,2022,pre,2022-11-18,Ban on Beer Is Latest Flash Point in World Cup...
2,NYT_nyt://article/4c666dfe-8e36-594e-867f-1b85...,random,NYT-main,2022,tournament,2022-11-23,Even the Saudi Team Is Stunned After Victory O...
3,NYP_nyp_20221117_ab6a7e0238,random,New York Post,2022,pre,2022-11-17,Floating hotels and $2K-a-night tents: Where W...
4,NYP_nyp_20221121_5d07e10b6b,purposeful,New York Post,2022,tournament,2022-11-21,FIFA threats force World Cup teams to abandon ...
5,NYP_nyp_20221124_095ac95416,random,New York Post,2022,tournament,2022-11-24,USMNT foolishly switching to prevent-defense s...
6,ATH_3895347,random,The Athletic,2022,pre,2022-11-14,Cristiano Ronaldo ‘focused’ on World Cup as Ma...
7,ATH_3940030,purposeful,The Athletic,2022,tournament,2022-11-27,U.S. Soccer take down social media posts after...
8,ATH_3966763,random,The Athletic,2022,tournament,2022-12-04,"The cult of Lionel Messi: From the pitch, to B..."
9,NYT_nyt://article/ade9c476-fd38-5872-b87d-2d5a...,purposeful,NYT-main,2026,pre,2026-06-05,Iran’s Soccer Team Allowed Into U.S. for World...


### Build cross-outlet candidates


In [10]:
# Build cross-outlet candidate pairs for each anchor and keep same-edition articles published within ±7 days
anchor_meta = anchors[["article_id", "anchor_type", "outlet", "edition", "pub_date", "headline_text"]].rename(columns={"article_id": "anchor_id", "outlet": "anchor_outlet", "edition": "anchor_edition", "pub_date": "anchor_date", "headline_text": "anchor_headline"})
candidate_meta = df[["article_id", "outlet", "edition", "pub_date", "headline_text"]].rename(columns={"article_id": "candidate_id", "outlet": "candidate_outlet", "edition": "candidate_edition", "pub_date": "candidate_date", "headline_text": "candidate_headline"})
pairs = anchor_meta.merge(candidate_meta, how="cross")
pairs["day_gap"] = (pairs["anchor_date"] - pairs["candidate_date"]).abs().dt.days
pairs = pairs[pairs["anchor_edition"].eq(pairs["candidate_edition"]) & pairs["anchor_outlet"].ne(pairs["candidate_outlet"]) & pairs["day_gap"].le(max(WINDOWS))].copy()

print("Candidate pairs within ±7 days:", len(pairs))
print(pairs["day_gap"].value_counts().sort_index())

Candidate pairs within ±7 days: 7245
day_gap
0     549
1     975
2     980
3    1001
4     987
5     880
6     919
7     954
Name: count, dtype: int64


### Spec A and Spec A′: headline token-set Jaccard similarity


In [11]:
def jaccard(a, b): return len(a & b) / len(a | b) if a | b else 0.0

pairs["jaccard"] = [jaccard(headline_sets[a], headline_sets[b]) for a, b in zip(pairs["anchor_id"], pairs["candidate_id"])]
pairs["jaccard_no_generic"] = [jaccard(headline_sets[a] - GENERIC_TERMS, headline_sets[b] - GENERIC_TERMS) for a, b in zip(pairs["anchor_id"], pairs["candidate_id"])]
print(pairs[["jaccard", "jaccard_no_generic"]].describe().round(3))


        jaccard  jaccard_no_generic
count  7245.000            7245.000
mean      0.084               0.009
std       0.067               0.028
min       0.000               0.000
25%       0.000               0.000
50%       0.105               0.000
75%       0.133               0.000
max       0.357               0.375


**Finding.** Generic tournament terms account for much of the raw headline overlap. Removing `world`, `cup`, `fifa`, `2022`, and `2026` reduces mean Jaccard similarity from 0.084 to 0.009 and the median from 0.105 to 0, suggesting that much of the baseline overlap comes from shared World Cup vocabulary rather than event-specific terms.

### Window experiment: Spec A nearest neighbors


In [12]:
results = []

for window in WINDOWS:
    eligible = pairs[pairs["day_gap"].le(window)].copy()
    ranked = eligible[eligible["jaccard"].gt(0)].sort_values(["anchor_id", "candidate_outlet", "jaccard", "day_gap", "candidate_id"], ascending=[True, True, False, True, True]).copy()
    ranked["rank"] = ranked.groupby(["anchor_id", "candidate_outlet"]).cumcount() + 1
    ranked["top1_tie_n"] = ranked.groupby(["anchor_id", "candidate_outlet"])["jaccard"].transform(lambda s: s.eq(s.iloc[0]).sum())
    ranked["cutoff_tie_excluded_n"] = ranked.groupby(["anchor_id", "candidate_outlet"])["jaccard"].transform(lambda s: s.iloc[TOP_K:].eq(s.iloc[TOP_K - 1]).sum() if len(s) > TOP_K else 0)
    ranked = ranked[ranked["rank"].le(TOP_K)].copy()
    ranked["window_days"] = window
    results.append(ranked)

neighbors = pd.concat(results, ignore_index=True)
print("Retrieved neighbors:", len(neighbors))
neighbors[["anchor_id", "anchor_outlet", "candidate_id", "candidate_outlet", "window_days", "rank", "jaccard", "day_gap", "top1_tie_n", "cutoff_tie_excluded_n"]].head(20)


Retrieved neighbors: 335


,anchor_id,anchor_outlet,candidate_id,candidate_outlet,window_days,rank,jaccard,day_gap,top1_tie_n,cutoff_tie_excluded_n
0,ATH_3895347,The Athletic,NYT_nyt://article/567a2248-2cb8-50fe-be56-9ee3...,NYT-main,1,1,0.125000,0,1,0
1,ATH_3895347,The Athletic,NYP_nyp_20221115_d793f9a70d,New York Post,1,1,0.142857,1,1,0
2,ATH_3895347,The Athletic,NYP_nyp_20221114_f34dec3622,New York Post,1,2,0.117647,0,1,0
3,ATH_3895347,The Athletic,NYP_nyp_20221114_edee08666a,New York Post,1,3,0.105263,0,1,0
4,ATH_3940030,The Athletic,NYT_nyt://article/3fa407e7-8140-587d-a232-540d...,NYT-main,1,1,0.133333,0,1,0
5,ATH_3940030,The Athletic,NYT_nyt://article/4cd4ab06-1284-54fd-80ce-1186...,NYT-main,1,2,0.062500,1,1,0
6,ATH_3940030,The Athletic,NYP_nyp_20221127_fd0f9d7d65,New York Post,1,1,0.250000,0,1,0
7,ATH_3940030,The Athletic,NYP_nyp_20221127_54b115bd91,New York Post,1,2,0.117647,0,1,0
8,ATH_3940030,The Athletic,NYP_nyp_20221128_03c17d564c,New York Post,1,3,0.105263,1,1,0
9,ATH_3966763,The Athletic,NYT_nyt://article/bf36283b-cc6d-50c3-84e6-4802...,NYT-main,1,1,0.071429,1,1,0


In [13]:
for window in WINDOWS:
    available = pairs[pairs["day_gap"].le(window)].groupby(["anchor_id", "candidate_outlet"])["jaccard"].max()
    print(f"±{window} day | anchor-target groups:", len(available), "| no nonzero neighbor:", available.eq(0).sum(), "| tied top-1:", neighbors[neighbors["window_days"].eq(window) & neighbors["rank"].eq(1)]["top1_tie_n"].gt(1).sum())


±1 day | anchor-target groups: 40 | no nonzero neighbor: 1 | tied top-1: 4
±3 day | anchor-target groups: 40 | no nonzero neighbor: 1 | tied top-1: 3
±7 day | anchor-target groups: 40 | no nonzero neighbor: 1 | tied top-1: 7


**Finding.** Spec A returns a nonzero neighbor for 39 of 40 anchor–target groups across all three windows. Top-1 ties remain limited, although they increase at ±7 days, suggesting that wider windows mainly introduce more competing candidates rather than solving missing-overlap cases.

### Spec B and Spec B′: term-frequency cosine similarity

In [14]:
# Apply the Stage 1 headline_lexical normalization to opening text for Spec B′, after dropping tweet URLs and @handles
def normalize_lexical(text):
    text = re.sub(r"\bU\.S\.?(?=\W|$)|\bUS\b", "usa", str(text))
    text = text.lower().replace("’", "'").replace("‘", "'")
    text = re.sub(r"'s\b", "", text).replace("'", "")
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    text = re.sub(r"(?<![a-z0-9])-+|-+(?![a-z0-9])", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df["opening_tokens"] = df["opening_250"].map(lambda text: [token for token in normalize_lexical(re.sub(r"pic\.twitter\.com/\S+|https?://\S+|@\w+", " ", text)).split() if token not in ENGLISH_STOP_WORDS])
headline_counts = df.set_index("article_id")["headline_tokens"].map(Counter).to_dict()
opening_counts = df.set_index("article_id")["opening_tokens"].map(Counter).to_dict()

print("Empty opening token lists:", df["opening_tokens"].map(len).eq(0).sum())
print()
print(pd.Series([token for tokens in df["opening_tokens"] for token in tokens]).value_counts().head(20))


Empty opening token lists: 0

world         11133
cup           10033
team           4217
said           3210
game           3126
tournament     2779
usa            2565
fifa           2330
players        2290
soccer         2215
england        2182
time           2169
match          2136
group          2108
just           1852
final          1721
new            1691
fans           1686
qatar          1685
stadium        1628
Name: count, dtype: int64


In [15]:
def tf_cosine(a, b):
    dot = sum(count * b[token] for token, count in a.items() if token in b)
    return dot / (sum(v * v for v in a.values()) * sum(v * v for v in b.values())) ** 0.5 if dot else 0.0

pairs["headline_cosine"] = [tf_cosine(headline_counts[a], headline_counts[b]) for a, b in zip(pairs["anchor_id"], pairs["candidate_id"])]
pairs["opening_cosine"] = [tf_cosine(opening_counts[a], opening_counts[b]) for a, b in zip(pairs["anchor_id"], pairs["candidate_id"])]
print(pairs[["jaccard", "headline_cosine", "opening_cosine"]].describe().round(3))


        jaccard  headline_cosine  opening_cosine
count  7245.000         7245.000        7245.000
mean      0.084            0.150           0.141
std       0.067            0.117           0.070
min       0.000            0.000           0.000
25%       0.000            0.000           0.091
50%       0.105            0.192           0.131
75%       0.133            0.236           0.180
max       0.357            0.537           0.617


**Finding.** Using the longer `opening_250` representation makes lexical overlap less sparse: opening-text cosine has a 25th percentile of 0.091, compared with 0 for headline Jaccard and headline cosine. This allows additional contextual language beyond the headline to influence similarity.

### Spec C: opening-text entity overlap

In [16]:
# Normalize entity strings: drop "the", possessives, and @; reduce people to last names; merge U.S. aliases; drop outlet self-references and generic World Cup / FIFA entities
def normalize_entity(text, label):
    text = re.sub(r"^the\s+|['’]s$|^@", "", text.lower().strip())
    if label == "PERSON": text = text.split()[-1] if text.split() else text
    return ENTITY_ALIASES.get(text, text)

nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])
df["entities"] = [sorted({normalize_entity(ent.text, ent.label_) for ent in doc.ents if ent.label_ in ENTITY_TYPES} - OUTLET_ENTITIES - {""}) for doc in nlp.pipe(df["opening_250"], batch_size=64)]
df["entities"] = df["entities"].map(lambda entities: [entity for entity in entities if "world cup" not in entity and entity != "fifa"])
entity_sets = df.set_index("article_id")["entities"].map(set).to_dict()

print("Empty entity sets:", df["entities"].map(len).eq(0).sum())
print()
print(pd.Series([entity for entities in df["entities"] for entity in entities]).value_counts().head(20))


Empty entity sets: 3

usa            1673
qatar           956
england         854
argentina       702
france          679
mexico          550
spain           503
brazil          463
messi           455
canada          428
germany         401
belgium         380
iran            332
paraguay        323
morocco         317
netherlands     293
portugal        254
pochettino      249
australia       240
los angeles     232
Name: count, dtype: int64


In [17]:
pairs["entity_jaccard"] = [jaccard(entity_sets[a], entity_sets[b]) for a, b in zip(pairs["anchor_id"], pairs["candidate_id"])]
print(pairs["entity_jaccard"].describe().round(3))

count    7245.000
mean        0.044
std         0.054
min         0.000
25%         0.000
50%         0.037
75%         0.062
max         0.455
Name: entity_jaccard, dtype: float64


**Finding.** Entity overlap is relatively sparse, with a mean Jaccard similarity of 0.044 and a median of 0.037. The most frequent retained entities are countries and teams such as the USA, Qatar, England, and Argentina, so this representation emphasizes shared actors and places rather than shared wording.

### Representation comparison at ±3 days

In [18]:
SPECS = {"A_headline_jaccard": "jaccard", "A_prime_headline_jaccard_no_generic": "jaccard_no_generic", "B_headline_tf_cosine": "headline_cosine", "B_prime_opening_tf_cosine": "opening_cosine", "C_opening_entity_jaccard": "entity_jaccard"}
eligible = pairs[pairs["day_gap"].le(COMPARISON_WINDOW)].copy()
spec_results = []

for spec, score_col in SPECS.items():
    ranked = eligible[eligible[score_col].gt(0)].sort_values(["anchor_id", "candidate_outlet", score_col, "day_gap", "candidate_id"], ascending=[True, True, False, True, True]).copy()
    ranked["rank"] = ranked.groupby(["anchor_id", "candidate_outlet"]).cumcount() + 1
    ranked["top1_tie_n"] = ranked.groupby(["anchor_id", "candidate_outlet"])[score_col].transform(lambda s: s.eq(s.iloc[0]).sum())
    ranked["cutoff_tie_excluded_n"] = ranked.groupby(["anchor_id", "candidate_outlet"])[score_col].transform(lambda s: s.iloc[TOP_K:].eq(s.iloc[TOP_K - 1]).sum() if len(s) > TOP_K else 0)
    ranked = ranked[ranked["rank"].le(TOP_K)].copy()
    ranked["spec"] = spec
    ranked["score"] = ranked[score_col]
    spec_results.append(ranked)

spec_neighbors = pd.concat(spec_results, ignore_index=True)
print("Retrieved neighbors:", len(spec_neighbors))
print(spec_neighbors["spec"].value_counts())


Retrieved neighbors: 546
spec
B_prime_opening_tf_cosine              118
C_opening_entity_jaccard               115
A_headline_jaccard                     113
B_headline_tf_cosine                   113
A_prime_headline_jaccard_no_generic     87
Name: count, dtype: int64


In [19]:
for spec, score_col in SPECS.items():
    available = eligible.groupby(["anchor_id", "candidate_outlet"])[score_col].max()
    print(f"{spec} | anchor-target groups:", len(available), "| no nonzero neighbor:", available.eq(0).sum(), "| tied top-1:", spec_neighbors[spec_neighbors["spec"].eq(spec) & spec_neighbors["rank"].eq(1)]["top1_tie_n"].gt(1).sum())


A_headline_jaccard | anchor-target groups: 40 | no nonzero neighbor: 1 | tied top-1: 3
A_prime_headline_jaccard_no_generic | anchor-target groups: 40 | no nonzero neighbor: 7 | tied top-1: 4
B_headline_tf_cosine | anchor-target groups: 40 | no nonzero neighbor: 1 | tied top-1: 2
B_prime_opening_tf_cosine | anchor-target groups: 40 | no nonzero neighbor: 0 | tied top-1: 0
C_opening_entity_jaccard | anchor-target groups: 40 | no nonzero neighbor: 1 | tied top-1: 5


In [20]:
# compare top-1 neighbors across specifications
top1 = spec_neighbors[spec_neighbors["rank"].eq(1)].set_index(["anchor_id", "candidate_outlet", "spec"])["candidate_id"].unstack("spec")
for spec in list(SPECS)[1:]: print(f"{spec} | top-1 same as Spec A:", top1[spec].eq(top1["A_headline_jaccard"]).sum(), "of", top1[spec].notna().sum())

A_prime_headline_jaccard_no_generic | top-1 same as Spec A: 24 of 33
B_headline_tf_cosine | top-1 same as Spec A: 33 of 39
B_prime_opening_tf_cosine | top-1 same as Spec A: 9 of 40
C_opening_entity_jaccard | top-1 same as Spec A: 4 of 39


**Finding.** Representation choice changes the retrieved comparison set substantially. Headline TF cosine selects the same top-1 neighbor as headline Jaccard in 33 of 39 available groups, while opening-text cosine and entity overlap agree with Spec A in only 9 of 40 and 4 of 39 groups, respectively.

In [21]:
top1_headlines = spec_neighbors[spec_neighbors["rank"].eq(1)].set_index(["anchor_headline", "candidate_outlet", "spec"])["candidate_headline"].unstack("spec")
top1_headlines

spec                                                                                                A_headline_jaccard  \
anchor_headline                                    candidate_outlet                                                      
Ban on Beer Is Latest Flash Point in World Cup ... New York Post     'Someone owes Budweiser $75 million': Experts ...   
                                                   The Athletic            Qatar bans beer sales at World Cup stadiums   
Christian Pulisic wastes no time ending five-mo... NYT-main                                                        NaN   
                                                   The Athletic      With goal drought over, Christian Pulisic can ...   
Cristiano Ronaldo ‘focused’ on World Cup as Man... NYT-main                             The World Cup’s Forgotten Team   
                                                   New York Post     All the controversies surrounding the Qatar Wo...   
Even the Saudi Team Is Stunned After Victory Ov... New York Post     Argentina stunned by Saudi Arabia in one of bi...   
                                                   The Athletic      Saudi Arabia humble Argentina – but was it the...   
FIFA threats force World Cup teams to abandon '... NYT-main          On Eve of World Cup, FIFA Chief Says, ‘Don’t C...   
                                                   The Athletic      OneLove armbands sell out after FIFA’s World C...   
Floating hotels and $2K-a-night tents: Where Wo... NYT-main                      The World Cup That Changed Everything   
                                                   The Athletic      What are fans allowed and not allowed to do wh...   
Iran’s Soccer Team Allowed Into U.S. for World ... New York Post     Pope Leo backs team USA in the World Cup — as ...   
                                                   The Athletic      Should we be calling it ‘soccer’ or ‘football’...   
MetLife World Cup train tickets confirmed at $1... NYT-main          World Cup Transit Prices Are Set: $150 by Trai...   
                                                   New York Post     NJ Transit plans to jack up train tickets from...   
Pressing World Cup Question for U.S.: Who’ll Sc... New York Post     Germany in crisis over bringing wives and girl...   
                                                   The Athletic      US Soccer seeking to join UEFA Working Group i...   
Somali Referee Says His World Cup Dream Is Dash... New York Post     FIFA prez Gianni Infantino says people 'should...   
                                                   The Athletic      Trump administration claim Somali World Cup re...   
Style points go a long way for USMNT in winning... NYT-main          Congolese Team’s World Cup Leopard Style Chann...   
                                                   New York Post     Bond between USMNT's Tim Weah and Weston McKen...   
The answer to the question Alexi Lalas gets the... NYT-main                                 A Very Different World Cup   
                                                   The Athletic      Tottenham’s Marcos Senesi gets last-minute Arg...   
The cult of Lionel Messi: From the pitch, to Bu... NYT-main          How a World Cup Podcaster (and Messi Superfan)...   
                                                   New York Post     Lionel Messi's first knockout-stage goal helps...   
Thomas Partey denied entry to Canada, unable to... NYT-main          Somali World Cup Referee Denied Entry to U.S. ...   
                                                   New York Post     Ghana star facing rape charges not allowed int...   
Trump Relents From Spain-Bashing to Present Its... New York Post     Trump seen marveling at World Cup trophy as he...   
                                                   The Athletic      Trump booed by World Cup final crowd, takes pa...   
U.S. Soccer take down social media posts after ... NYT-main          Call It Football or Soccer.

### Export top-1 pairs for manual labeling

### Manual labeling codebook

Labels were judged from `headline_text` and `opening_250`; full text was consulted only when unclear.

- **`same_event`**: both articles primarily cover the same **specific, bounded news episode or focal development**, such as the same match, decision or announcement, incident, statement, or controversy development. Different angles or genres are allowed. Immediate reactions, explanations, and analyses count when they remain organized around the same triggering development.

  A later, separately reported development — such as a new official decision, appeal ruling, lawsuit, later statement introducing a distinct development, institutional or corporate response, or other downstream action — is **not** the same event. If both developments belong to the same continuing storyline, the pair is coded `same_issue`.

  For match coverage, articles are `same_event` only when both primarily concern the **same specific fixture** (the same two teams in the same game). Previews, match reports, and post-match coverage count when the match itself — including its result, performances, tactics, injuries, cards, officiating, or other match incidents — is the organizing subject. Stories merely occurring around the same match, such as celebrity attendance, TV ratings, merchandise, fan-zone atmosphere, security, ticket prices, or unrelated political statements, are not `same_event` with match coverage.

- **`same_issue`**: the articles cover **different specific news episodes or focal developments** within the same clearly identifiable ongoing storyline or substantive issue. A shared person, team, country, organization, or broad World Cup topic is not sufficient by itself. Different matches involving the same team count as `same_issue` only when both explicitly connect to the same continuing storyline or issue. Multi-match roundups and coverage of different matches are therefore not automatically `same_issue`.

- **`not_related`**: the articles share neither the same bounded news episode nor a clearly identifiable ongoing storyline or substantive issue. Shared countries, teams, players, organizations, generic entities, or broad themes alone are insufficient.

First ask whether both articles primarily cover the same bounded news episode or focal development (`same_event`). If not, ask whether they cover different developments within the same clearly identifiable continuing storyline or substantive issue (`same_issue`). Otherwise, code `not_related`.

Note: A development mentioned only as background does not make a pair `same_event`. Publication or update timing alone also does not determine the label; coding is based on each article's primary organizing subject.

In [22]:
# Unique top-1 pairs across the window experiment and the representation comparison; spec, window, and score are hidden
top1_pairs = pd.concat([neighbors[neighbors["rank"].eq(1)], spec_neighbors[spec_neighbors["rank"].eq(1)]])[["anchor_id", "candidate_id"]].drop_duplicates()
assert not set(zip(top1_pairs["anchor_id"], top1_pairs["candidate_id"])) & set(zip(top1_pairs["candidate_id"], top1_pairs["anchor_id"])), "Reverse duplicate pairs found"
text = df.set_index("article_id")[["outlet", "pub_date", "headline_text", "opening_250"]]
to_label = top1_pairs.join(text.add_prefix("anchor_"), on="anchor_id").join(text.add_prefix("candidate_"), on="candidate_id").sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
to_label.insert(0, "pair_id", range(1, len(to_label) + 1))
to_label["label"] = ""
to_label["notes"] = ""

print("Unique top-1 pairs to label:", len(to_label))
to_label.to_csv(os.path.join(OUTPUT_DIR, "similarity_pairs_to_label.csv"), index=False, encoding="utf-8-sig")


Unique top-1 pairs to label: 129


### Save outputs


In [23]:
ANCHOR_COLS = ["article_id", "anchor_type", "outlet", "edition", "period", "pub_date", "url", "headline_text"]
anchors[ANCHOR_COLS].to_csv(os.path.join(OUTPUT_DIR, "similarity_anchors.csv"), index=False)
neighbors.to_csv(os.path.join(OUTPUT_DIR, "similarity_specA_jaccard_neighbors.csv"), index=False)
spec_neighbors.to_csv(os.path.join(OUTPUT_DIR, "similarity_spec_comparison_neighbors.csv"), index=False)

print("Saved.")

Saved.


### Manual label evaluation

In [24]:
LABELS = ["same_event", "same_issue", "not_related"]
labels = pd.read_csv(LABELED_FILE)[["anchor_id", "candidate_id", "label"]]

assert labels["label"].isin(LABELS).all(), f"Every pair needs one of: {LABELS}"
assert not labels.duplicated(["anchor_id", "candidate_id"]).any()
assert set(zip(to_label["anchor_id"], to_label["candidate_id"])) <= set(zip(labels["anchor_id"], labels["candidate_id"]))

print(labels["label"].value_counts())

label
not_related    72
same_event     34
same_issue     23
Name: count, dtype: int64


In [25]:
# Evaluate Spec A top-1 matches across candidate windows using the manual labels
window_top1 = neighbors[neighbors["rank"].eq(1)].merge(labels, on=["anchor_id", "candidate_id"], how="left")
window_top1["same_event"] = window_top1["label"].eq("same_event")

# Inspect manual-label distributions by anchor type and window size
print(pd.crosstab([window_top1["anchor_type"], window_top1["window_days"]], window_top1["label"]))
print()

# There are 16 possible purposeful and 24 possible random anchor-target groups (two target outlets per anchor)
EXPECTED_GROUPS = {"purposeful": 16, "random": 24}
window_summary = (
    window_top1.groupby(["anchor_type", "window_days"])
    .agg(same_event_n=("same_event", "sum"), returned_top1=("candidate_id", "size"))
    .reset_index()
)
window_summary["expected_groups"] = window_summary["anchor_type"].map(EXPECTED_GROUPS)
window_summary["coverage"] = window_summary["returned_top1"] / window_summary["expected_groups"]
window_summary["same_event_rate_returned"] = window_summary["same_event_n"] / window_summary["returned_top1"]
window_summary["same_event_rate_all_groups"] = window_summary["same_event_n"] / window_summary["expected_groups"]
print(window_summary.round(2).to_string(index=False))

# Primary known-positive diagnostic: restrict evaluation to the 12 purposeful groups with a manually confirmed same-event counterpart
# All 12 confirmed counterparts fall within ±1 day, so the denominator remains 12 across all three windows
UNCONFIRMED_PURPOSEFUL_GROUPS = {
    (PURPOSEFUL_IDS[1], "NYT-main"),       # A2 -> NYT
    (PURPOSEFUL_IDS[4], "New York Post"),  # A5 -> NYP
    (PURPOSEFUL_IDS[5], "NYT-main"),       # A6 -> NYT
    (PURPOSEFUL_IDS[6], "NYT-main"),       # A7 -> NYT
}
all_purposeful_groups = {
    (row.article_id, target_outlet)
    for row in purposeful_anchors.itertuples()
    for target_outlet in df["outlet"].drop_duplicates()
    if target_outlet != row.outlet
}
CONFIRMED_POSITIVE_GROUPS = all_purposeful_groups - UNCONFIRMED_PURPOSEFUL_GROUPS
assert len(CONFIRMED_POSITIVE_GROUPS) == 12

# Calculate top-1 recovery within the confirmed-positive groups
confirmed_window = window_top1[[(a, o) in CONFIRMED_POSITIVE_GROUPS for a, o in zip(window_top1["anchor_id"], window_top1["candidate_outlet"])]]
confirmed_window_summary = (
    confirmed_window.groupby("window_days")
    .agg(same_event_n=("same_event", "sum"), returned_top1=("candidate_id", "size"))
    .reset_index()
)
confirmed_window_summary["confirmed_groups"] = 12
confirmed_window_summary["coverage"] = confirmed_window_summary["returned_top1"] / 12
confirmed_window_summary["known_positive_top1_rate"] = confirmed_window_summary["same_event_n"] / 12

print()
print("Confirmed-positive purposeful groups:")
print(confirmed_window_summary.round(2).to_string(index=False))



label                    not_related  same_event  same_issue
anchor_type window_days                                     
purposeful  1                      3          12           1
            3                      4           8           4
            7                      5           8           3
random      1                     14           7           2
            3                     17           6           0
            7                     17           3           3

anchor_type  window_days  same_event_n  returned_top1  expected_groups  coverage  same_event_rate_returned  same_event_rate_all_groups
 purposeful            1            12             16               16      1.00                      0.75                        0.75
 purposeful            3             8             16               16      1.00                      0.50                        0.50
 purposeful            7             8             16               16      1.00                      0.50

In [26]:
# Identify Spec A top-1 matches that change when widening the candidate window beyond ±1 day
first = window_top1[window_top1["window_days"].eq(1)][["anchor_id", "candidate_outlet", "candidate_id"]].rename(columns={"candidate_id": "top1_at_1"})
wider = window_top1[window_top1["window_days"].gt(1)].merge(first, on=["anchor_id", "candidate_outlet"], how="left")
entrants = wider[wider["candidate_id"].ne(wider["top1_at_1"])]

print(pd.crosstab(entrants["window_days"], entrants["label"])) # check

label        not_related  same_event  same_issue
window_days                                     
3                      9           1           4
7                     16           1           6


In [27]:
# Compare top-1 retrieval labels across similarity specifications at the fixed ±3-day window
spec_top1 = spec_neighbors[spec_neighbors["rank"].eq(1)].merge(labels, on=["anchor_id", "candidate_id"], how="left")
spec_top1["same_event"] = spec_top1["label"].eq("same_event")

# Inspect manual-label distributions by anchor type and specification
print(pd.crosstab([spec_top1["anchor_type"], spec_top1["spec"]], spec_top1["label"]))
print()

# summarize retrieval coverage and same-event rates for all anchor-target groups
spec_summary = (
    spec_top1.groupby(["anchor_type", "spec"])
    .agg(same_event_n=("same_event", "sum"), returned_top1=("candidate_id", "size"))
    .reset_index()
)

spec_summary["expected_groups"] = spec_summary["anchor_type"].map(EXPECTED_GROUPS)
spec_summary["coverage"] = spec_summary["returned_top1"] / spec_summary["expected_groups"]
spec_summary["same_event_rate_returned"] = spec_summary["same_event_n"] / spec_summary["returned_top1"]
spec_summary["same_event_rate_all_groups"] = spec_summary["same_event_n"] / spec_summary["expected_groups"]
print(spec_summary.round(2).to_string(index=False))


label                                            not_related  same_event  \
anchor_type spec                                                           
purposeful  A_headline_jaccard                             4           8   
            A_prime_headline_jaccard_no_generic            2          10   
            B_headline_tf_cosine                           5           8   
            B_prime_opening_tf_cosine                      3           8   
            C_opening_entity_jaccard                       9           4   
random      A_headline_jaccard                            17           6   
            A_prime_headline_jaccard_no_generic           10           6   
            B_headline_tf_cosine                          17           6   
            B_prime_opening_tf_cosine                     11           9   
            C_opening_entity_jaccard                      12           7   

label                                            same_issue  
anchor_type spec         

In [28]:
# Primary known-positive comparison: evaluate each specification on the same 12 confirmed-positive purposeful groups
confirmed_spec = spec_top1[[(a, o) in CONFIRMED_POSITIVE_GROUPS for a, o in zip(spec_top1["anchor_id"], spec_top1["candidate_outlet"])]]
confirmed_spec_summary = (
    confirmed_spec.groupby("spec")
    .agg(same_event_n=("same_event", "sum"), returned_top1=("candidate_id", "size"))
    .reset_index()
)
confirmed_spec_summary["confirmed_groups"] = 12
confirmed_spec_summary["coverage"] = confirmed_spec_summary["returned_top1"] / 12
confirmed_spec_summary["known_positive_top1_rate"] = confirmed_spec_summary["same_event_n"] / 12

# Report known-positive recovery and descriptive same-event rates by target outlet
print()
print("Confirmed-positive purposeful groups:")
print(confirmed_spec_summary.round(2).to_string(index=False))
print()
print(spec_top1.pivot_table(index="spec", columns="candidate_outlet", values="same_event", aggfunc="mean").round(2))


Confirmed-positive purposeful groups:
                               spec  same_event_n  returned_top1  confirmed_groups  coverage  known_positive_top1_rate
                 A_headline_jaccard             8             12                12       1.0                      0.67
A_prime_headline_jaccard_no_generic            10             12                12       1.0                      0.83
               B_headline_tf_cosine             8             12                12       1.0                      0.67
          B_prime_opening_tf_cosine             8             12                12       1.0                      0.67
           C_opening_entity_jaccard             4             12                12       1.0                      0.33

candidate_outlet                     NYT-main  New York Post  The Athletic
spec                                                                      
A_headline_jaccard                       0.08           0.50          0.46
A_prime_headline_jacc

### Results

The purposeful anchors are the primary known-positive retrieval diagnostic. Among the 12 anchor–target groups for which full-text review confirmed that a same-event counterpart exists, Spec A retrieves the correct event in 12/12 groups at ±1 day, compared with 8/12 at ±3 days and 8/12 at ±7 days. The wider windows add mostly `same_issue` or `not_related` top-1 neighbors rather than additional exact-event matches, so the narrower window is descriptively better for this lexical baseline.

At the fixed ±3-day representation comparison, **A′** (headline Jaccard with generic tournament terms removed) has the highest observed known-positive top-1 rate: **10/12**, compared with 8/12 for A, 8/12 for B, 8/12 for B′, and 4/12 for C. A′ retains full coverage on the 12 confirmed-positive groups, although it returns a nonzero top-1 for only 17/24 random anchor–target groups. Thus, its apparent improvement should be interpreted together with coverage. Headline Jaccard and headline TF cosine behave similarly, while opening-text cosine changes the retrieved neighbor more substantially and gives the highest same-event count among the random anchors (9/24).

Entity overlap is weaker on the confirmed-positive diagnostic, although performance varies by target outlet. These differences are descriptive because most specification contrasts amount to only one or two groups in a 12-group known-positive sample.

The stratified-random anchors provide a naturalistic robustness check rather than an accuracy estimate, because the existence of a same-event counterpart was not established for every random anchor–target group.

### Experimentation, failures, revisions, and next steps

- **Experiment 1 — Generic-term removal (Spec A vs. A′).** I first used headline-token Jaccard similarity with standard stop-word removal (Spec A), then tested a revised representation (A′) that additionally removed corpus-generic terms such as `world`, `cup`, `fifa`, `2022`, and `2026`. The baseline showed that these frequent tournament terms could create lexical overlap without identifying the same event. A′ improved retrieval among the confirmed-positive purposeful groups but reduced coverage among the random anchors, revealing a trade-off between specificity and retaining potentially useful signal.

- **Experiment 2 — Headline representation and additional context (Specs A, B, and B′).** I compared headline Jaccard (A) with headline term-frequency cosine similarity (B). The two often returned similar neighbors because short headlines contain few repeated words, making their representations close to binary. I then added B′, using the first 250 words of the article, to test whether more context could recover same-event articles with differently worded headlines. B′ produced more `same_event` matches among the random anchor–target groups (9/24 versus 6/24 for A and B), but it did not improve retrieval on the 12 confirmed-positive groups at ±3 days. This suggests that added context can help with paraphrasing but does not consistently improve strict event matching.

- **Experiment 3 and failure — Candidate-window sensitivity (±1, ±3, ±7 days).** Holding Spec A fixed, I varied only the publication-date window, expecting wider windows to recover delayed cross-outlet reporting. This did not happen: on the 12 confirmed-positive groups, Spec A recovered the same event in 12/12 groups at ±1 day but only 8/12 at ±3 and ±7 days. Of the 14 top-1 matches that changed when widening to ±3 days, only one was `same_event` (9 `not_related`, 4 `same_issue`). This shows that expanding the candidate set mainly introduced articles about the same actors or broader issues but different developments, reducing event-level precision.

- **Experiment 4 — Entity-based similarity (Spec C).** I tested whether named-entity overlap could identify comparable articles when outlets used different vocabulary. Entity matching sometimes captured shared actors or organizations that lexical methods missed, but it also introduced new problems: recurring actors could create false similarity, and normalization remained incomplete for related forms such as `Iran` and `Iranian`. Spec C recovered 4/12 of the confirmed-positive groups, the lowest of the five specifications. I therefore treat entity overlap as a complementary representation rather than a stand-alone event-matching method.

- **Challenge and revision — Text preprocessing.** Earlier data preprocessing revealed several issues that could distort similarity, including inconsistent apostrophes, outlet-specific text artifacts, and the risk of treating “U.S.” as the stop word `us`. These problems were corrected before the Week 1 similarity analysis so that the comparison would reflect substantive text differences rather than preprocessing artifacts.

- **Challenge and revision — Evaluation design.** The purposeful anchors provide useful known-positive tests but are intentionally easier than ordinary corpus cases because cross-outlet coverage was expected and candidate counterparts were verified in advance. Full-text review confirmed a same-event counterpart for 12 of the 16 purposeful anchor–target groups. I therefore use these 12 groups as the primary known-positive retrieval diagnostic and analyze the stratified-random anchors separately as a naturalistic robustness check rather than treating either set as a general accuracy estimate.

- **Challenge and revision — Manual labeling rules.** Before labeling began, I refined the definitions of `same_event`, `same_issue`, and `not_related` to distinguish a bounded news episode from later developments in the same storyline and to clarify how match coverage should be treated. The final codebook was fixed before I computed any label summaries by specification or window. Labels were assigned without specification, window, or similarity score being shown.

- **Next steps.** For later thesis matching, I will repeat the representation comparison at ±1 day, where the window experiment performed best, and use additional human review to validate the final matching procedure. Week 2 will separately evaluate TF-IDF and embedding-based semantic representations rather than extending the Week 1 lexical comparison.

### Social and cultural reflection

The Week 1 results clarify an important measurement problem for the thesis: **“similar coverage” is not a neutral or self-evident category.** Before comparing framing across NYT-main, the New York Post, and The Athletic, I need to decide which articles are comparable enough that differences in wording are not simply differences in the underlying event. This echoes the Week 1 lecture’s framing of similarity choices as measurement decisions: the key question is not only which metric to use, but what features matter and what kind of similarity is being measured. In the Week 1 reading, Gentzkow and Shapiro (2010) make a related measurement choice by constructing their media-slant measure from selected phrase frequencies and excluding phrases that are unlikely to be diagnostically useful. My A′ result points in the same direction: generic World Cup vocabulary can create surface overlap without necessarily improving event identification.

The time-window experiment reinforces this point. The narrower ±1-day window recovered more of the confirmed same-event counterparts than the wider windows, showing that expanding the comparison set can introduce articles that share teams, actors, or tournament vocabulary while describing different developments. This also matters substantively because cross-outlet attention is selective: failure to retrieve a same-event counterpart may reflect a limitation of the matching method, but it may also reflect a real composition difference in what an outlet chose to cover. In this sense, differences across outlets can arise not only from how the same event is presented, but also from which events receive attention in the first place. The stratified-random anchors provide a useful naturalistic check on retrieval behavior, but their same-event rates are not accuracy estimates because counterpart existence was not established for every group.

The representation comparison also suggests that headline-based similarity may be sensitive to outlet-specific headline style, although the evidence is descriptive and based on a small diagnostic sample. Among returned top-1 matches when NYT is the target outlet, the same-event rate is 0.08 for A and B, compared with 0.23 for B′ and 0.25 for C. Inspection shows that some NYT headlines are relatively indirect or feature-style, which may make simple headline overlap less effective at identifying substantively comparable articles. However, these differences could also reflect the particular events represented in the anchor sample, so they should not be interpreted as a general outlet-level effect.

Most importantly, **same-event similarity is not framing similarity**. My later SQ2 dimensions—problem definition, responsibility attribution, evaluation, and treatment or response—closely map onto Entman’s (1993) framing functions of defining problems, diagnosing causes, making moral evaluations, and suggesting remedies. In the Somali-referee episode, the NYT headline centers the referee whose World Cup “dream is dashed,” the New York Post foregrounds Infantino saying people “should chill,” and The Athletic foregrounds the Trump administration’s claim of suspected “terror” ties. This comparison is based only on the headlines, but it illustrates how reports about the same bounded development can emphasize different affected actors, responsibility claims, evaluations, and responses. Similarity therefore helps construct a defensible set of comparable cross-outlet articles, but it does not itself measure framing. Week 1 establishes which articles can reasonably be compared; the later framing analysis is still needed to answer SQ2.

**References**

Entman, R. M. (1993). Framing: Toward clarification of a fractured paradigm. *Journal of Communication, 43*(4), 51–58.

Gentzkow, M., & Shapiro, J. M. (2010). What drives media slant? Evidence from U.S. daily newspapers. *Econometrica, 78*(1), 35–71.

### Analysis Summary

- The window experiment varies only the candidate window (±1, ±3, ±7 days) for Spec A; the representation comparison fixes the window at ±3 days and compares Specs A, A′, B, B′, and C.
- Candidates must come from the same World Cup edition and a different outlet, with retrieval conducted separately for each target outlet. Ties are broken by smaller date distance and then `article_id`; zero-similarity candidates are excluded.
- The sample includes 8 purposeful anchors and a stratified-random set across edition, outlet, and tournament period, yielding 16 purposeful and 24 random anchor–target groups.
- Full-text verification identified same-event counterparts for 12 of the 16 purposeful groups. These 12 groups are the primary retrieval diagnostic; random anchors are analyzed separately as a naturalistic robustness check rather than an accuracy estimate.
- Manual evaluation uses `same_event`, `same_issue`, and `not_related`. Unique top-1 pairs are labeled without specification, window, or score being shown; complete blinding to anchor type is not possible because the purposeful anchors were researcher-selected.

### Data note

The full `worldcup_method_corpus.parquet` is not included because it contains copyrighted full-text news articles. The submitted materials therefore include the analysis code and derived Week 1 outputs, while the source corpus is saved locally.